# NASA Milling Wear Dataset — Deep Audit (Phase 1)

**Author:** Bright (Implementation: AntiGravity Agent | Strategic Lead: ChatGPT)
**Dataset:** NASA Prognostics Center of Excellence (PCoE) / UC Berkeley BEST Lab (Agogino & Goebel, 1999)
**Objective:** Systematic technical audit of raw milling experimental data before formulating the ML problem for V2.

In [ ]:
import os
import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib.pyplot as plt
from pathlib import Path

DATA_PATH = Path('../data/mill.mat')
print(f"Checking dataset at: {DATA_PATH.resolve()}")
assert DATA_PATH.exists(), "mill.mat not found. Please run download_dataset.py first."

mat = sio.loadmat(str(DATA_PATH))
mill = mat['mill']
print(f"Loaded 'mill' structured array: shape={mill.shape}, total runs={mill.shape[1]}")

## 1. Extract Struct Metadata & Inspect Variables

In [ ]:
records = []
signal_names = ['smcAC', 'smcDC', 'vib_table', 'vib_spindle', 'AE_table', 'AE_spindle']

for i in range(mill.shape[1]):
    item = mill[0, i]
    case_val = int(item['case'][0, 0])
    run_val = int(item['run'][0, 0])
    vb_raw = item['VB']
    vb_val = float(vb_raw[0, 0]) if vb_raw.size > 0 else np.nan
    time_val = float(item['time'][0, 0]) if item['time'].size > 0 else np.nan
    doc_val = float(item['DOC'][0, 0]) if item['DOC'].size > 0 else np.nan
    feed_val = float(item['feed'][0, 0]) if item['feed'].size > 0 else np.nan
    mat_val = int(item['material'][0, 0]) if item['material'].size > 0 else np.nan
    
    sig_lens = {f"{sf}_len": item[sf].size for sf in signal_names}
    
    records.append({
        'run_idx': i + 1,
        'case': case_val,
        'run': run_val,
        'VB': vb_val,
        'time': time_val,
        'DOC': doc_val,
        'feed': feed_val,
        'material': mat_val,
        **sig_lens
    })

df = pd.DataFrame(records)
print(f"Total runs: {len(df)}")
print(f"Unique cases (tool inserts): {df['case'].nunique()}")
print(f"Runs with measured VB: {df['VB'].notna().sum()} / {len(df)}")
print(f"Runs with missing VB (NaN): {df['VB'].isna().sum()} / {len(df)}")
df.head(10)

## 2. Experimental Cases Summary (Operating Conditions & Tool Life)

In [ ]:
case_summary = df.groupby('case').agg(
    total_runs=('run', 'count'),
    vb_measured=('VB', lambda x: x.notna().sum()),
    vb_min=('VB', 'min'),
    vb_max=('VB', 'max'),
    DOC=('DOC', 'first'),
    feed=('feed', 'first'),
    material=('material', lambda x: 'Cast Iron (1)' if x.iloc[0]==1 else 'Steel J45 (2)')
).reset_index()
case_summary

## 3. Tool Wear (VB) Progression Across All 16 Cases

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16, 12), sharey=True)
axes = axes.flatten()

for c_id in range(1, 17):
    ax = axes[c_id - 1]
    sub = df[df['case'] == c_id]
    doc = sub['DOC'].iloc[0]
    feed = sub['feed'].iloc[0]
    mat_name = 'Cast Iron' if sub['material'].iloc[0] == 1 else 'Steel'
    
    ax.plot(sub['run'], sub['VB'], 'o-', color='#1f77b4', markersize=4, label='VB (mm)')
    ax.axhline(0.3, color='r', linestyle='--', alpha=0.7, label='ISO threshold (0.3mm)')
    ax.set_title(f"Case {c_id}: {doc}mm, {feed}mm/r, {mat_name}", fontsize=9)
    ax.set_xlabel("Pass / Run #", fontsize=8)
    ax.set_ylabel("VB (mm)", fontsize=8)
    ax.grid(True, alpha=0.3)
    if c_id == 1:
        ax.legend(fontsize=7)

plt.suptitle("Flank Wear (VB) vs Run Number Across 16 Experimental Tool Inserts", fontsize=13, y=0.99)
plt.tight_layout()
plt.show()

## 4. Raw Sensor Signal Inspection (Fresh vs Worn Tool)

In [ ]:
# Compare Case 1 Run 1 (Fresh, VB=0.0) vs Case 1 Run 17 (Worn, VB=0.50)
idx_fresh = df[(df['case'] == 1) & (df['run'] == 1)].index[0]
idx_worn = df[(df['case'] == 1) & (df['run'] == 17)].index[0]

fresh_item = mill[0, idx_fresh]
worn_item = mill[0, idx_worn]

time_axis = np.arange(9000) / 250.0 # 250 Hz -> seconds

fig, axes = plt.subplots(6, 2, figsize=(14, 12), sharex=True)
for row, sf in enumerate(signal_names):
    axes[row, 0].plot(time_axis, fresh_item[sf].flatten()[:9000], color='#2ecc71', lw=0.6)
    axes[row, 0].set_ylabel(sf, fontsize=8)
    if row == 0: axes[row, 0].set_title("Fresh Tool (Case 1, Run 1, VB=0.0mm)")
    
    axes[row, 1].plot(time_axis, worn_item[sf].flatten()[:9000], color='#e74c3c', lw=0.6)
    if row == 0: axes[row, 1].set_title("Worn Tool (Case 1, Run 17, VB=0.50mm)")
    
axes[5, 0].set_xlabel("Time (s)")
axes[5, 1].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()

## 5. Summary of Data Leakage & Time Structure Findings

1. **Definite Leakage:** None in the sensor stream, but `run` index and `time` within a case correlate with cumulative tool use. If `run` is fed to a model, the model predicts the clock, not the physics.
2. **Group Structure:** 16 independent tool inserts (Cases). Splitting data randomly by runs causes severe cross-validation leakage across the same tool's wear trajectory. **Leave-One-Tool-Out (GroupKFold by Case)** or **Leave-Operating-Condition-Out** is mandatory.
3. **Missing Targets:** 21 of 167 runs lack VB labels. Only 146 runs can be used for supervised regression without interpolation.